In [ ]:
from pathlib import Path
import re

import pandas as pd
import mercury as mr


LANGUAGE_PATTERNS = {
    "JavaScript / TypeScript": r"\bjavascript\b|\btypescript\b|\bnode\.?js\b|\breact\b",
    "Python": r"\bpython\b",
    "Ruby": r"\bruby\b|\brails\b",
    "Go": (
        r"(?i:\bgolang\b)|"
        r"\bGo\b(?=\s*(?:[,/|+&;)]|developer|engineer|backend|experience|stack|programming))"
    ),
    "Java": r"\bjava\b",
    "Rust": r"\brust\b",
    "PHP": r"\bphp\b",
    "Scala": r"\bscala\b",
}

CASE_SENSITIVE_LANGUAGES = {"Go"}

LANGUAGE_COLORS = {
    "JavaScript / TypeScript": "#f7df1e",
    "Python": "#3776ab",
    "Ruby": "#cc342d",
    "Go": "#00add8",
    "Java": "#e76f00",
    "Rust": "#8b5e3c",
    "PHP": "#777bb4",
    "Scala": "#dc322f",
}

MIN_POST_SHARE_PCT = 2

data_roots = [Path.cwd(), Path.cwd() / "who-is-hiring"]
data_dir = next(
    (root for root in data_roots if list(root.glob("who_is_hiring_[0-9][0-9][0-9][0-9].csv.gz"))),
    None,
)
if data_dir is None:
    raise FileNotFoundError("Could not find the yearly Who is Hiring CSV files.")

year_files = {
    int(match.group(1)): path
    for path in data_dir.glob("who_is_hiring_*.csv.gz")
    if (match := re.fullmatch(r"who_is_hiring_(\d{4})\.csv\.gz", path.name))
}
available_years = sorted(year_files)

In [ ]:
anchor_years = [year for year in [2012, 2018, 2026] if year in year_files]
year_filter = mr.MultiSelect(
    label="Years to compare",
    value=[str(year) for year in anchor_years],
    choices=[str(year) for year in available_years],
    placeholder="Select 2–5 years",
    url_key="years",
    key="sankey-years",
)

In [ ]:
selected_years = sorted(int(year) for year in year_filter.value)
if not selected_years:
    _ = mr.Markdown("Select at least one year to build the Sankey diagram.")
    mr.Stop()

yearly_jobs = []
for year in selected_years:
    year_jobs = pd.read_csv(year_files[year])
    year_jobs["year"] = year
    yearly_jobs.append(year_jobs)

jobs = pd.concat(yearly_jobs, ignore_index=True)
jobs["search_text"] = jobs["comment"].fillna("").astype(str)

In [ ]:
language_filter = mr.MultiSelect(
    label="Languages",
    value=list(LANGUAGE_PATTERNS),
    choices=list(LANGUAGE_PATTERNS),
    placeholder="Select languages",
    url_key="languages",
    key="sankey-languages",
)

In [ ]:
metric_filter = mr.Select(
    label="Ribbon metric",
    value="Share of tracked mentions (%)",
    choices=["Share of tracked mentions (%)", "Post mentions"],
    url_key="metric",
    key="sankey-metric",
)

In [ ]:
if len(selected_years) == 1:
    period_label = f"in {selected_years[0]}"
else:
    period_label = " vs ".join(str(year) for year in selected_years)

_ = mr.Markdown(
    f"# Programming language mentions {period_label}\n\n"
    "Compare the language mix in monthly Ask HN ‘Who is hiring?’ posts. "
    "Each ribbon connects a year to a language mentioned in that year’s posts.",
    key="sankey-title",
)

In [ ]:
selected_languages = list(language_filter.value)
if not selected_languages:
    _ = mr.Markdown("Select at least one language to build the Sankey diagram.")
    mr.Stop()

total_posts = jobs.groupby("year")["comment_id"].nunique()
mention_frames = []
for language in selected_languages:
    mask = jobs["search_text"].str.contains(
        LANGUAGE_PATTERNS[language],
        case=language in CASE_SENSITIVE_LANGUAGES,
        regex=True,
        na=False,
    )
    language_mentions = jobs.loc[mask, ["year", "comment_id"]].copy()
    language_mentions["language"] = language
    mention_frames.append(language_mentions)

mentions = pd.concat(mention_frames, ignore_index=True)
counts = (
    mentions.groupby(["year", "language"], as_index=False)["comment_id"]
    .nunique()
    .rename(columns={"comment_id": "posts"})
)
all_pairs = pd.MultiIndex.from_product(
    [selected_years, selected_languages], names=["year", "language"]
)
counts = (
    counts.set_index(["year", "language"])
    .reindex(all_pairs, fill_value=0)
    .reset_index()
)
counts["post_share_pct"] = counts.apply(
    lambda row: 100 * row["posts"] / total_posts.loc[row["year"]], axis=1
)
yearly_mentions = counts.groupby("year")["posts"].transform("sum")
counts["mention_share_pct"] = (
    100 * counts["posts"] / yearly_mentions.where(yearly_mentions.ne(0), 1)
)
displayed_counts = counts[
    counts["post_share_pct"].ge(MIN_POST_SHARE_PCT) & counts["posts"].gt(0)
].copy()

value_column = (
    "mention_share_pct"
    if metric_filter.value == "Share of tracked mentions (%)"
    else "posts"
)
flows = displayed_counts.rename(
    columns={"year": "source", "language": "target", value_column: "value"}
)[["source", "target", "value"]]
flows["source"] = flows["source"].astype(str)

In [ ]:
posts_analyzed = int(total_posts.sum())
posts_with_language = mentions["comment_id"].nunique()
total_language_mentions = int(counts["posts"].sum())
language_totals = counts.groupby("language")["posts"].sum()
leading_language = language_totals.idxmax() if not language_totals.empty else "—"

mr.Indicator([
    mr.Indicator(f"{posts_analyzed:,}", label="Posts analyzed"),
    mr.Indicator(f"{posts_with_language:,}", label="Posts with a tracked language"),
    mr.Indicator(f"{total_language_mentions:,}", label="Language mentions"),
    mr.Indicator(leading_language, label="Most-mentioned language"),
])

## Language mix by year

In [ ]:
if flows.empty:
    _ = mr.Markdown(
        f"No selected language appears in at least {MIN_POST_SHARE_PCT}% of posts."
    )
else:
    sankey_chart = mr.Sankey(
        flows,
        source="source",
        target="target",
        value="value",
        colors=LANGUAGE_COLORS,
        height=max(480, 105 * len(selected_years)),
        show_values=True,
        value_format=".1f" if value_column == "mention_share_pct" else ",",
    )
    sankey_chart.display()

In [ ]:
metric_explanation = (
    "Ribbon values are percentages of all selected-language mentions within each year. "
    f"Before languages mentioned in fewer than {MIN_POST_SHARE_PCT}% of posts are hidden, "
    "these values sum to 100% per year."
    if value_column == "mention_share_pct"
    else "Ribbon values are counts of posts mentioning each language."
)
_ = mr.Markdown(metric_explanation, key="sankey-metric-note")

## Displayed language flows

In [ ]:
summary_table = displayed_counts.rename(columns={
    "year": "Year",
    "language": "Language",
    "posts": "Posts",
    "post_share_pct": "Share of posts (%)",
    "mention_share_pct": "Share of tracked mentions (%)",
})
summary_table["Share of posts (%)"] = summary_table["Share of posts (%)"].round(1)
summary_table["Share of tracked mentions (%)"] = (
    summary_table["Share of tracked mentions (%)"].round(1)
)
summary_table = summary_table.sort_values(["Year", "Posts"], ascending=[True, False])

_ = mr.Table(
    summary_table,
    page_size=25,
    search=True,
    height="500px",
    key="language-flow-summary",
)

In [ ]:
partial_year_note = (
    " The 2026 data is a partial-year snapshot." if 2026 in selected_years else ""
)
_ = mr.Markdown(
    f"""## Methodology and limitations

Each language is counted at most once per post, but one post may mention multiple languages. Therefore, the independent **Share of posts (%)** values can sum to more than 100%. The default Sankey metric instead divides each language count by all selected-language mentions in its year, producing an additive composition suitable for ribbon widths. Languages mentioned in fewer than **{MIN_POST_SHARE_PCT}% of posts** in a year are hidden to keep the diagram readable.

The diagram represents **year → language mention** relationships. It does not imply that Ruby jobs became Go jobs or that the same vacancies were tracked through time. Keyword matching is case-insensitive except where capitalization helps distinguish the Go language from the English verb. React and Node.js are included in JavaScript / TypeScript; Rails is included in Ruby.{partial_year_note}

The local dataset contains comments from monthly [Ask HN: Who is hiring?](https://news.ycombinator.com/) threads. Built with [Mercury's Sankey widget](https://runmercury.com/docs/output/sankey/).""",
    key="sankey-methodology",
)